# 02 — Decision Logic Walkthrough
Step through the decision engine: signals, behavioral controls, and explainable outputs.

In [ ]:
import sys; sys.path.insert(0,'..')
import warnings; warnings.filterwarnings('ignore')
from src.data_ingestion import ingest_city_data
from src.preprocessing import clean, aggregate_time_windows, compute_rolling_baseline, city_level_aggregate
from src.signal_extraction import extract_all_signals
from src.decision_engine import DecisionConfig, evaluate_all_signals, decisions_to_dataframe
print('Ready')

In [ ]:
raw=ingest_city_data('nyc',days_back=45,force_synthetic=True)
cl=clean(raw); agg=aggregate_time_windows(cl,freq='h')
base=compute_rolling_baseline(agg,freq='h'); city_agg=city_level_aggregate(base,freq='h')
signals=extract_all_signals(city_agg,base,stride=6)
print(f'Extracted {len(signals)} signals')

## Default policy

In [ ]:
results,engine=evaluate_all_signals(signals)
print('Action counts:',engine.memory.action_counts())
df=decisions_to_dataframe(results)
df[df['action']=='ACT'][['category','activity_ratio','z_score','trend','confidence']].head(8)

## Naive vs Conservative

In [ ]:
configs={'naive':DecisionConfig(act_ratio_threshold=1.2,z_score_threshold=1.0,required_confirmations=1,cooldown_hours=0),'default':DecisionConfig(),'conservative':DecisionConfig.strict()}
for name,cfg in configs.items():
    _,eng=evaluate_all_signals(signals,config=cfg)
    c=eng.memory.action_counts(); total=sum(c.values()) or 1
    print(f'{name:15s}  ACT={c.get("ACT",0):4d}  rate={c.get("ACT",0)/total:.1%}  FAR={eng.memory.false_alert_rate_estimate():.1%}')

## A full ACT decision

In [ ]:
acts=[(s,d) for s,d in results if d.action=='ACT']
if acts: print(acts[0][1].display())